# 01 — Zero-shot baseline: Qwen2.5-0.5B-Instruct on GSM8K test + SVAMP

Input: output of `00_data` (*Add Input* → that notebook). Greedy decoding, one model copy per GPU.

Set `DEBUG = True` first (16 problems per dataset), check the log, then `False` for the full run.

In [ ]:
DEBUG = True

In [ ]:
import json, logging, os, re, sys, time
from concurrent.futures import ThreadPoolExecutor
from glob import glob
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

CFG = dict(
    run_name="exp1_zeroshot_0.5b",
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=512,
    batch_size=32,
    limit=16 if DEBUG else None,   # problems per dataset
    debug=DEBUG,
)

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    found = glob("/kaggle/input/**/gsm8k_test.jsonl", recursive=True)
    assert found, "add the 00_data notebook as an input"
    DATA_DIR, OUT_DIR = Path(found[0]).parent, Path("/kaggle/working") / CFG["run_name"]
else:
    DATA_DIR, OUT_DIR = Path("../outputs/data"), Path("../outputs") / CFG["run_name"]
if DEBUG:
    OUT_DIR = OUT_DIR.with_name(OUT_DIR.name + "_debug")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Logging
Every line goes to the notebook and to `OUT_DIR/run.log`. `main` = notebook, `rank0` / `rank1` = the worker on each GPU.

In [ ]:
# root at WARNING hides library chatter (HTTP requests etc.); our own loggers log at INFO
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s | %(name)-5s | %(message)s", datefmt="%H:%M:%S", force=True,
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "run.log")],
)
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    return logger

log = get_logger("main")

def fmt_time(s):
    return f"{int(s // 60)}m{int(s % 60):02d}s"

log.info(f"config: {CFG}")
log.info(f"data: {DATA_DIR} | out: {OUT_DIR}")

## Data

In [ ]:
data = {
    "gsm8k": pd.read_json(DATA_DIR / "gsm8k_test.jsonl", lines=True, dtype={"gold": str}),
    "svamp": pd.read_json(DATA_DIR / "svamp_test.jsonl", lines=True, dtype={"gold": str}),
}
if CFG["limit"]:
    data = {k: v.head(CFG["limit"]) for k, v in data.items()}
log.info(f"problems: { {k: len(v) for k, v in data.items()} }")

## Prompt + answer checking

Shared by every later notebook (eval and rewards), so keep them identical everywhere.

- `extract_pred`: number after the last `####`; if the model skipped `####`, fall back to the last number in the text. `has_format` records which case happened.
- `is_correct`: numeric compare, so `18`, `18.0`, `18.00` all match.

In [ ]:
INSTRUCTION = "Solve the problem step by step. End your answer with a final line of the form '#### <number>'."

def build_prompt(tok, question):
    msgs = [{"role": "user", "content": f"{question}\n\n{INSTRUCTION}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")

def extract_pred(text):
    if "####" in text:
        nums = NUM_RE.findall(text.split("####")[-1])
        if nums:
            return nums[0].replace(",", ""), True
    nums = NUM_RE.findall(text)
    return (nums[-1].replace(",", ""), False) if nums else (None, False)

def is_correct(pred, gold):
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

assert extract_pred("so 1,000 total\n#### 1,000") == ("1000", True)
assert extract_pred("#### $18.") == ("18", True)
assert extract_pred("#### -3") == ("-3", True)
assert extract_pred("#### 2.5 hours") == ("2.5", True)
assert extract_pred("3 + 4 = 7, so the answer is 72") == ("72", False)
assert extract_pred("no idea") == (None, False)
assert is_correct("18.00", "18") and not is_correct("17", "18") and not is_correct(None, "18")

## Models — one copy per GPU

In [ ]:
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
dtype = torch.float16 if devices[0].startswith("cuda") else torch.float32   # T4: no bf16
log.info(f"devices: {devices} | dtype: {dtype} | torch {torch.__version__} | transformers {transformers.__version__}")

tok = AutoTokenizer.from_pretrained(CFG["model_id"], padding_side="left")
models = []
for d in devices:
    t0 = time.time()
    models.append(AutoModelForCausalLM.from_pretrained(CFG["model_id"], dtype=dtype).to(d).eval())
    log.info(f"model loaded on {d} in {time.time() - t0:.1f}s")

print(build_prompt(tok, "What is 2 + 3?"))

## Generation

Prompts are sorted by length (less padding) and dealt round-robin to the GPUs. Each GPU runs in its own thread; GPU work doesn't hold Python's lock, so both run at the same time.

In [ ]:
@torch.inference_mode()
def generate_batch(model, prompts):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, do_sample=False, temperature=None, top_p=None, top_k=None,
                         max_new_tokens=CFG["max_new_tokens"], pad_token_id=tok.pad_token_id)
    new = out[:, enc.input_ids.shape[1]:]                     # drop the prompt tokens
    n_tokens = (new != tok.pad_token_id).sum(dim=1).tolist()
    return tok.batch_decode(new, skip_special_tokens=True), n_tokens


def run_rank(rank, shard, name):
    rlog, model, bs = get_logger(f"rank{rank}"), models[rank], CFG["batch_size"]
    n_batches = (len(shard) + bs - 1) // bs
    rlog.info(f"{name} | start on {model.device} | {len(shard)} problems, {n_batches} batches")
    rows, n_correct, t0 = [], 0, time.time()
    try:
        for b in range(n_batches):
            chunk, tb = shard[b * bs:(b + 1) * bs], time.time()
            texts, n_tokens = generate_batch(model, [p for _, p, _ in chunk])
            for (idx, _, gold), text, n in zip(chunk, texts, n_tokens):
                pred, has_format = extract_pred(text)
                ok = is_correct(pred, gold)
                n_correct += ok
                rows.append(dict(idx=idx, response=text, pred=pred, has_format=has_format, correct=ok,
                                 n_tokens=n, truncated=n >= CFG["max_new_tokens"]))
            done, elapsed = len(rows), time.time() - t0
            rlog.info(f"{name} | batch {b + 1}/{n_batches} | {done}/{len(shard)} done | acc so far {n_correct / done:.3f} | "
                      f"{sum(n_tokens) / (time.time() - tb):.0f} tok/s | elapsed {fmt_time(elapsed)} | "
                      f"ETA {fmt_time(elapsed / done * (len(shard) - done))}")
            n_trunc = sum(n >= CFG["max_new_tokens"] for n in n_tokens)
            if n_trunc > len(chunk) // 4:
                rlog.warning(f"{name} | batch {b + 1}: {n_trunc}/{len(chunk)} responses hit max_new_tokens")
    except Exception:
        rlog.exception(f"{name} | failed at batch {b + 1}/{n_batches}")
        raise
    rlog.info(f"{name} | finished {len(shard)} problems in {fmt_time(time.time() - t0)} | acc {n_correct / len(shard):.3f}")
    return rows


def evaluate(name, df):
    df = df.reset_index(drop=True)
    prompts = [build_prompt(tok, q) for q in df.question]
    order = sorted(range(len(df)), key=lambda i: len(prompts[i]), reverse=True)
    shards = [[(i, prompts[i], df.gold[i]) for i in order[r::len(models)]] for r in range(len(models))]
    log.info(f"{name} | start | {len(df)} problems on {len(models)} device(s)")
    t0 = time.time()
    with ThreadPoolExecutor(len(models)) as pool:
        parts = list(pool.map(run_rank, range(len(models)), shards, [name] * len(models)))
    wall = time.time() - t0
    res = pd.DataFrame([r for part in parts for r in part]).set_index("idx").sort_index()
    log.info(f"{name} | done in {fmt_time(wall)}")
    return df.join(res), wall

## Run

In [ ]:
results, metrics = {}, {}
for name, df in data.items():
    out, wall = evaluate(name, df)
    results[name] = out
    metrics[name] = dict(
        n=len(out),
        accuracy=round(float(out.correct.mean()), 4),
        format_rate=round(float(out.has_format.mean()), 4),
        mean_tokens=round(float(out.n_tokens.mean()), 1),
        truncated_rate=round(float(out.truncated.mean()), 4),
        wall_min=round(wall / 60, 2),
        gpu_hours=round(wall * len(models) / 3600, 3) if devices[0].startswith("cuda") else 0.0,
    )
    log.info(f"{name} | metrics: {metrics[name]}")

pd.DataFrame(metrics).T

## A few wrong answers

In [ ]:
for name, out in results.items():
    for _, r in out[~out.correct].head(3).iterrows():
        print(f"=== {name} | gold={r.gold} | pred={r.pred} | format={r.has_format} | tokens={r.n_tokens}")
        print(r.question, "\n--- response ---\n" + r.response + "\n")

## Save

In [ ]:
for name, out in results.items():
    out.to_json(OUT_DIR / f"predictions_{name}.jsonl", orient="records", lines=True)
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUT_DIR / "config.json").write_text(json.dumps(
    {**CFG, "instruction": INSTRUCTION, "devices": devices, "dtype": str(dtype),
     "torch": torch.__version__, "transformers": transformers.__version__}, indent=2))
log.info(f"saved: {sorted(p.name for p in OUT_DIR.iterdir())}")